In [1]:
from google.colab import drive, runtime
drive.mount('/content/drive', force_remount=True)
import os, sys, glob, json
sys.path.insert(0, '/content/drive/MyDrive/repos/deep-learning')

EXPERIMENT = '014_segformer_polyp'
REPO = 'deep-learning'

print('=' * 60)
print(f'🧪 E2E TEST: {EXPERIMENT}')
print('=' * 60)

Mounted at /content/drive
🧪 E2E TEST: 014_segformer_polyp


In [2]:
data_path = '/content/drive/MyDrive/data_lake/01_bronze_medical/cvc_datasetninja/images'
assert os.path.exists(data_path), f'❌ Data not found: {data_path}'
items = os.listdir(data_path)
assert len(items) > 0, f'❌ Data directory is empty'
print(f'✅ [1/5] Data verified: {len(items)} items')

✅ [1/5] Data verified: 612 items


In [3]:
exp_path = f'/content/drive/MyDrive/repos/{REPO}/experiments/{EXPERIMENT}'
required = ['config.yaml', 'train.py', 'README.md']
for f in required:
    assert os.path.exists(os.path.join(exp_path, f)), f'❌ Missing: {f}'
    print(f'  📄 {f}')
print(f'✅ [2/5] All experiment files present')

  📄 config.yaml
  📄 train.py
  📄 README.md
✅ [2/5] All experiment files present


In [4]:
model_patterns = [
    f'/content/drive/MyDrive/repos/{REPO}/experiments/{EXPERIMENT}/**/*.pt',
]
found = []
for p in model_patterns:
    found.extend(glob.glob(p, recursive=True))
assert len(found) > 0, '❌ No trained model found'
for m in found[:5]:
    print(f'  📦 {m}')
print(f'✅ [3/5] Trained model: {len(found)} file(s)')

  📦 /content/drive/MyDrive/repos/deep-learning/experiments/014_segformer_polyp/checkpoints/best_model.pt
  📦 /content/drive/MyDrive/repos/deep-learning/experiments/014_segformer_polyp/checkpoints/best_model_epoch0.pt
  📦 /content/drive/MyDrive/repos/deep-learning/experiments/014_segformer_polyp/checkpoints/best_model_epoch1.pt
  📦 /content/drive/MyDrive/repos/deep-learning/experiments/014_segformer_polyp/checkpoints/best_model_epoch2.pt
  📦 /content/drive/MyDrive/repos/deep-learning/experiments/014_segformer_polyp/checkpoints/best_model_epoch3.pt
✅ [3/5] Trained model: 11 file(s)


In [5]:
!pip install -q mlflow transformers albumentations
import mlflow
mlflow.set_tracking_uri('file:///content/drive/MyDrive/mlflow/mlruns')
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], max_results=5)
assert len(runs) > 0, f'❌ No MLflow runs for {EXPERIMENT}'
best = runs.iloc[0]
for col in runs.columns:
    if col.startswith('metrics.'):
        print(f'  📊 {col.replace("metrics.", "")}: {best[col]}')
print(f'✅ [4/5] MLflow run: {best["run_id"][:8]}...')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 11.5 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


  📊 train_dice: 0.9464591896533966
  📊 lr: 1e-06
  📊 test_iou: 0.7877937264931507
  📊 train_loss: 0.27808295071125033
  📊 val_iou: 0.8415220288129953
  📊 test_dice: 0.8678299433145767
  📊 val_loss: 0.41580978494424087
  📊 val_dice: 0.9018137363287119
✅ [4/5] MLflow run: 3b883aaf...


In [6]:
import torch
from transformers import SegformerForSemanticSegmentation
import yaml

config_path = os.path.join(exp_path, 'config.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

best_model_path = glob.glob(f'{exp_path}/checkpoints/best_model_*.pt')[0]

model = SegformerForSemanticSegmentation.from_pretrained(
    config['model']['architecture'],
    num_labels=1,
    ignore_mismatched_sizes=True
)

checkpoint = torch.load(best_model_path, map_location='cpu')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
with torch.no_grad():
    dummy = torch.randn(1, 3, 352, 352)
    output = model(dummy)
    print(f'  Output shape: {output.logits.shape}')
print(f'✅ [5/5] Inference test passed')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/99.0M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/364 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/98.9M [00:00<?, ?B/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b2
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.bias                               | UNEXPECTED | 
classifier.weight                             | UNEXPECTED | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.running_mean           | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.linear_fuse.weight                | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.classifier.bias                   | MISSING    | 
decode_head.classifier.weight                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different ta

  Output shape: torch.Size([1, 1, 88, 88])
✅ [5/5] Inference test passed


In [7]:
print('\n' + '=' * 60)
print('🏁 E2E TEST COMPLETE')
print('=' * 60)
checks = [
    'Data exists and is non-empty',
    'Experiment files are complete',
    'Trained model checkpoint exists',
    'MLflow tracking has logged runs',
    'Model inference produces valid output',
]
for i, c in enumerate(checks, 1):
    print(f'  ✅ {i}. {c}')
print(f'\n🎉 ALL {len(checks)} CHECKS PASSED')
print('=' * 60)
runtime.unassign()


🏁 E2E TEST COMPLETE
  ✅ 1. Data exists and is non-empty
  ✅ 2. Experiment files are complete
  ✅ 3. Trained model checkpoint exists
  ✅ 4. MLflow tracking has logged runs
  ✅ 5. Model inference produces valid output

🎉 ALL 5 CHECKS PASSED
